# Replication: the same grid at seed 43

**Requires a GPU.** Produced `data/curves_seed43.csv`.

Identical protocol to `03_grid_experiment.ipynb` with a different seed, plus a
probe that adds a third episode length when memory allows, and a low
learning-rate arm (1e-6 with an L2-SP anchor at alpha = 0.003, compared at matched
weight displacement rather than matched steps) that answers the obvious
objection to a negative result measured at 1e-5.

*Body in French. Provenance only - the analysis notebook reads the CSV.*

---


# Taille d'épisode x taille de contexte — l'asymétrie## Avant de lancer : attache la sortie du run précédent**Add Input → Notebook Output → le run `contexte_x_finetuning_v1`.**Ce notebook relit `curves.csv` et `manifest.json` de ce run, vérifie que le protocoleest identique, et **ne recalcule pas ce qui existe déjà** : la courbe de base des 24datasets, la grille complète à épisodes de 128 lignes, et les épisodes de 640 lignessur 4 datasets. Sans cette pièce jointe le notebook tourne quand même, mais il repartde zéro et il n'ira pas au bout en une session.## Ce qu'on chercheTon run précédent a montré deux choses. La première est solide : le gain du fine-tuningdécroît avec la taille du contexte et change de signe vers 300-500 lignes (23 datasetssur 24, test des signes p < 10⁻⁵). La seconde, sur 4 datasets seulement, est plusintéressante et pas encore établie : **allonger les épisodes d'entraînement déplace cecroisement vers la droite.**Deux hypothèses rendent compte de cette seconde observation, et elles ne se distinguent**pas** sur la diagonale :- **Correspondance** — il faut entraîner à la taille exacte du contexte de déploiement.- **Asymétrie** — il faut entraîner sur des épisodes *au moins aussi longs* que le  contexte de déploiement ; descendre en dessous ne coûte rien, monter au-dessus casse.Tes 4 datasets penchent déjà pour l'asymétrie : les épisodes de 640 lignes gagnent aussià 128 et à 256 lignes de contexte, là où la correspondance voudrait qu'ils perdent.Mais 4 datasets, des écarts de 0,005 et aucun test : ce n'est pas un résultat.## Le dispositifLa **grille complète** : chaque taille d'épisode évaluée à **toutes** les tailles decontexte. C'est le hors-diagonale qui tranche entre les deux hypothèses ; ne mesurer quela diagonale produirait des chiffres que les deux hypothèses prédisent à l'identique.| Épisodes | Statut | Datasets éligibles ||:--|:--|:--|| 128 lignes (102 + 26) | déjà mesuré sur 24 | tous || 640 lignes (512 + 128) | mesuré sur 4, à compléter | ceux dont le train ≥ 800 || une troisième taille, plus grande | à mesurer | déterminée par un prétest |La troisième taille n'est pas choisie d'avance : un prétest mesure ce qui tient enmémoire sur la T4 **et** ce qui tient dans le budget de temps restant, puis retient laplus grande des deux contraintes. Il imprime son arithmétique.Un dataset n'est éligible à une taille d'épisode que si son train peut fournir unépisode entier avec de la marge. Toutes les comparaisons entre tailles se font donc surle **sous-ensemble commun**, jamais sur des colonnes de composition différente — c'estla réserve principale du run précédent, et elle est traitée ici.## La quantité qui devrait tout expliquerSi l'asymétrie est la bonne lecture, alors le gain ne dépend pas séparément du contexteet de l'épisode, mais de leur **rapport** : contexte de déploiement ÷ contexted'entraînement. Les trois courbes devraient alors **se superposer** quand on les tracecontre ce rapport, avec la cassure à 1. C'est un effondrement de données, et c'est uneprédiction beaucoup plus exigeante qu'une simple décroissance : trois courbes quitombent l'une sur l'autre ne s'obtiennent pas par hasard.Si elles ne se superposent pas, l'asymétrie est fausse et il faut le dire.**Coût : 3 à 5 h** avec la pièce jointe, selon la troisième taille retenue. Une garde detemps coupe proprement entre deux datasets.

In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','tabpfn==9.0.0',
                'openml>=0.15,<0.17','scikit-learn>=1.5,<1.8','scipy','matplotlib','nbformat>=5'],check=True)
print('Dépendances installées. Redémarrer le noyau si Kaggle le demande, puis relancer tout.')

In [ ]:

import os
os.environ['TABPFN_NO_BROWSER']='1'
if not os.environ.get('TABPFN_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception as exc:
        raise RuntimeError('Créer et autoriser le secret Kaggle TABPFN_TOKEN (clé API Prior Labs).') from exc
    if not token: raise RuntimeError('Le secret TABPFN_TOKEN est vide.')
    os.environ['TABPFN_TOKEN']=token.strip(); del token
print('Authentification chargée en mémoire, clé masquée.')

## 1. Configuration

In [ ]:

from pathlib import Path

SEED = 42
SUITE_ID = 457
ALL_TASK_IDS = [363621,363626,363619,363623,363682,363684,
                363613,363618,363624,363627,363629,363632,
                363671,363674,363676,363679,363681,363689,
                363691,363694,363696,363700,363706,363712]
N_SHARDS, SHARD_ID = 1, 0

# ---- axe 1 : le contexte d'inférence (identique au run précédent, sinon rien
#      n'est réutilisable) --------------------------------------------------
CONTEXT_SIZES     = (32,64,128,256,512,1024,2048,None)
REF_CONTEXT       = 1024
SMALL_CONTEXT_MAX = 256
REPEATS_SMALL     = 3

# ---- axe 2 : la taille des épisodes d'entraînement ------------------------
# (contexte, requêtes) ; le rapport de requêtes reste 0,2 partout.
EPISODE_FIXED      = ((102,26),(512,128))              # 128 et 640 lignes
EPISODE_CANDIDATES = ((1638,410),(1229,307),(819,205)) # 2048, 1536, 1024 lignes
MIN_ELIGIBLE       = 10      # une taille d'épisode sous ce seuil ne vaut pas le calcul
ELIG_SLACK         = 1.25    # train >= 1,25 x épisode : marge pour la stratification

FT_MILESTONES = (25,50,200)
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP = 1e-5, 0.01, 1.0

# ---- axe 3 : l'agressivité de l'optimisation -----------------------------
# Real-TabPFN (arXiv 2507.03971) obtient des GAINS avec 3e-7, decay nul et un
# ancrage L2-SP vers les poids d'origine. C'est l'objection la plus prévisible à
# un résultat négatif obtenu à 1e-5 : « vous ne mesurez pas le fine-tuning, vous
# mesurez un pas trop grand ». Cette phase y répond.
# Le budget de pas est augmenté pour que le DÉPLACEMENT des poids soit du même
# ordre : la comparaison se fait à déplacement apparié, pas à pas apparié.
RUN_LR_PHASE   = True
LR_LOW         = 1e-6        # 10x plus petit ; 3e-7 en 200 pas ne bougerait rien
LR_LOW_WD      = 0.0         # le decay vers zéro est remplacé par l'ancrage
L2SP_ALPHA     = 0.003       # valeur de Real-TabPFN
LR_MILESTONES  = (250,1000)
LR_EPISODE     = (512,128)   # taille d'épisode intermédiaire, déjà couverte
LR_PHASE_TASKS = 4

# ---- évaluation ----------------------------------------------------------
REPORT_EVAL_CAP = 1024
PRED_BATCHES    = (64,16,4,1)

# ---- import du run précédent --------------------------------------------
IMPORT_PREVIOUS   = True
IMPORT_TOLERANCE  = 5e-3   # au-delà, l'import entier est rejeté, pas rafistolé
IMPORT_CHECK_N    = 3      # datasets re-mesurés pour vérifier

# ---- garde de temps ------------------------------------------------------
TIME_BUDGET_HOURS = 8.0
CURVE_SECONDS_GUESS = 120.0   # coût d'une courbe complète, affiné dès la 1re mesurée

# OpenML renvoie regulierement des 503 : on reessaie au lieu de perdre le run.
OPENML_RETRIES, OPENML_BACKOFF = 6, 10

SKIP_ON_ERROR = True
MIN_TARGETS = 12
MAX_DATASET_ROWS, MAX_FEATURES = 50_000, 110
MIN_FREE_GIB = 3.0
LOCAL_TRAIN_CAP = REF_CONTEXT

RUN_TAG = 'episode_x_contexte_v1'
ON_KAGGLE = Path('/kaggle/working').exists()
ROOT    = (Path('/kaggle/working') if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG
SCRATCH = (Path('/kaggle/temp')   if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG
ROOT.mkdir(parents=True,exist_ok=True); SCRATCH.mkdir(parents=True,exist_ok=True)

SHARD_IDS = ALL_TASK_IDS[SHARD_ID::N_SHARDS]
assert len(ALL_TASK_IDS)==len(set(ALL_TASK_IDS))==24
assert REF_CONTEXT in CONTEXT_SIZES and None in CONTEXT_SIZES
assert list(FT_MILESTONES)==sorted(set(FT_MILESTONES))
assert all(a>b for (a,_),(b,_) in zip(EPISODE_CANDIDATES,EPISODE_CANDIDATES[1:])), \
       'candidats à ranger du plus grand au plus petit'
assert EPISODE_CANDIDATES[-1][0]>EPISODE_FIXED[-1][0], 'la 3e taille doit dépasser les deux acquises'
assert LR_LOW<LEARNING_RATE and max(LR_MILESTONES)*LR_LOW>=0.4*max(FT_MILESTONES)*LEARNING_RATE, \
       'budget de pas trop faible pour atteindre un déplacement comparable'

def arm_name(ep_rows,steps,regime='standard'):
    return (f'ep{int(ep_rows)}_{int(steps)}' if regime=='standard'
            else f'{regime}_ep{int(ep_rows)}_{int(steps)}')

print('Résultats :',ROOT)
print('Contextes :',CONTEXT_SIZES)
print('Épisodes acquis :',[a+b for a,b in EPISODE_FIXED],'lignes')
print('Candidats pour le 3e :',[a+b for a,b in EPISODE_CANDIDATES],'lignes (prétest)')
print('Budgets FT :',FT_MILESTONES,'pas | garde de temps',TIME_BUDGET_HOURS,'h')

## 2. Imports et utilitaires

In [ ]:

import re, gc, json, time, random, shutil, hashlib, inspect, traceback, importlib.metadata
import numpy as np, pandas as pd, torch, openml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from scipy.stats import spearmanr
from IPython.display import display

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from tabpfn.base import get_embeddings as tabpfn_get_embeddings
from tabpfn.model_loading import save_tabpfn_model
from tabpfn.finetuning.finetuned_classifier import FinetunedTabPFNClassifier
from tabpfn.finetuning.finetuned_base import _parameters_unused_by_task
from tabpfn.finetuning.data_util import get_preprocessed_dataset_chunks, meta_dataset_collator
from tabpfn.finetuning._torch_compat import GradScaler, autocast, sdpa_kernel_context
from tabpfn.architectures.interface import PerformanceOptions

assert importlib.metadata.version('tabpfn')=='9.0.0', 'tabpfn != 9.0.0 : redémarrer le noyau après pip.'
assert torch.cuda.is_available(), 'Activer le GPU (Settings -> Accelerator -> GPU).'
assert hasattr(ModelVersion,'V3_5')
assert 'shuffle' in inspect.signature(get_preprocessed_dataset_chunks).parameters
DEVICE='cuda:0'; GIB=2**30
print('GPU :',torch.cuda.get_device_name(0),'/',round(torch.cuda.get_device_properties(0).total_memory/GIB,2),'Gio')

def clean_gpu():
    gc.collect(); torch.cuda.empty_cache()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sha(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    tmp.replace(p)

def write_csv(path,frame):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(p)

def check_disk(label,where=None,extra=0.0):
    free=shutil.disk_usage(where or SCRATCH).free/GIB
    if free < MIN_FREE_GIB+extra:
        raise RuntimeError(f'{label} : {free:.1f} Gio libres. Exporter les CSV puis relancer un shard plus petit.')

def choose_indices(y,n,seed):
    # Sous-échantillonnage stratifié déterministe, >= 2 exemples par classe.
    y=np.asarray(y); n=min(int(n),len(y))
    classes,counts=np.unique(y,return_counts=True)
    if len(classes)!=2 or min(counts)<2: raise ValueError('Deux classes avec >= 2 lignes requises')
    if n==len(y): return np.arange(len(y))
    if n<4: raise ValueError('Échantillon trop petit')
    k1=max(2,min(int(counts[1]),int(round(n*counts[1]/len(y))))); k0=n-k1
    if k0>counts[0]: k0=int(counts[0]); k1=n-k0
    if k0<2: k0=2; k1=n-k0
    if k1>counts[1] or k0>counts[0] or min(k0,k1)<2: raise ValueError('Stratification impossible')
    rng=np.random.default_rng(int(seed))
    a=rng.choice(np.flatnonzero(y==classes[0]),k0,replace=False)
    b=rng.choice(np.flatnonzero(y==classes[1]),k1,replace=False)
    return np.sort(np.r_[a,b])

def scores(y,p):
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any(): raise ValueError('Probabilités invalides')
    if len(np.unique(y))!=2: raise ValueError('Métrique indéfinie sur un split mono-classe')
    return dict(auc=float(roc_auc_score(y,p)),
                logloss=float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),
                ap=float(average_precision_score(y,p)))

def is_oom(e):
    return isinstance(e,torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()

FAILURES=[]
def record_failure(phase,tid,name,exc,**extra):
    # Un échec est écarté du run, jamais masqué : il apparaît dans failures.csv et à l écran.
    row=dict(phase=phase,task_id=tid,name=name,error=type(exc).__name__,
             message=str(exc)[:400],**extra)
    FAILURES.append(row)
    p=ROOT/'failures.csv'
    tmp=p.with_name(p.name+'.tmp'); pd.DataFrame(FAILURES).to_csv(tmp,index=False); tmp.replace(p)
    print(f'  !! ÉCARTÉ [{phase}] {name} ({tid}) — {type(exc).__name__}: {str(exc)[:160]}',flush=True)
    if not SKIP_ON_ERROR: raise exc
    return row

def openml_retry(fn,*a,**kw):
    # OpenML renvoie regulierement des 503, ou coupe en cours de route. On reessaie
    # avec attente croissante plutot que de perdre un run de plusieurs heures.
    last=None
    for k in range(OPENML_RETRIES):
        try: return fn(*a,**kw)
        except Exception as exc:
            last=exc
            if k==OPENML_RETRIES-1: break
            wait=min(OPENML_BACKOFF*(2**k),120)
            print(f'    OpenML indisponible ({type(exc).__name__}) - nouvelle tentative '
                  f'dans {wait}s [{k+1}/{OPENML_RETRIES-1}]',flush=True)
            time.sleep(wait)
    raise last

set_seed(SEED)
print('Utilitaires prêts.')

## 3. Métadonnées et splits**Rigoureusement identiques au run précédent** : mêmes identifiants, même graine, mêmedécoupage. C'est la condition pour que les mesures importées soient comparables auxnouvelles — et le prétest d'import le vérifie au lieu de le supposer.

In [ ]:

openml.config.set_root_cache_directory(str(SCRATCH/'openml_cache'))
check_disk('métadonnées OpenML')
# Appartenance a la suite : un controle utile, mais pas au prix du run.
# Les 24 identifiants sont figes et deja valides dans les runs precedents.
try:
    suite=openml_retry(openml.study.get_suite,SUITE_ID)
    allowed=set(map(int,suite.tasks))
    assert set(ALL_TASK_IDS)<=allowed
    print('Suite 457 verifiee.')
except Exception as exc:
    print(f'Suite 457 inverifiable ({type(exc).__name__}) - on poursuit avec les '
          '24 identifiants figes.',flush=True)

def clean_description(s):
    s=re.sub(r'<[^>]+>',' ',str(s or ''))
    s=re.sub(r'https?://\S+',' ',s)
    return re.sub(r'\s+',' ',s).strip()[:2500]

META={}; TASKS={}; TEXT={}; seen=set()
for tid in list(ALL_TASK_IDS):
    try:
        t=openml_retry(openml.tasks.get_task,tid,download_data=False,download_qualities=False)
        ds=openml_retry(t.get_dataset,download_data=False)
        did=int(t.dataset_id)
        if did in seen: raise ValueError(f'{tid}: doublon dataset_id OpenML')
        txt=f'{ds.name}. Prediction target {t.target_name}. '+clean_description(ds.description)
        if len(txt)<20: raise ValueError(f'{tid}: description trop courte pour la baseline textuelle')
        seen.add(did)
        META[tid]={'task_id':tid,'dataset_id':did,'name':str(ds.name),'target':str(t.target_name),
                   'description':txt,'url':f'https://www.openml.org/t/{tid}'}
        TASKS[tid]=t; TEXT[tid]=txt
        print(tid,META[tid]['name'])
    except Exception as exc:
        record_failure('metadata',tid,f'task_{tid}',exc)
ALL_TASK_IDS=[t for t in ALL_TASK_IDS if t in TASKS]
if len(ALL_TASK_IDS)<MIN_TARGETS:
    raise RuntimeError(f'{len(ALL_TASK_IDS)} tâches seulement : voir failures.csv.')
print(f'{len(ALL_TASK_IDS)}/24 tâches OpenML disponibles.')
write_json(ROOT/'catalog_meta.json',META)

In [ ]:

def encode_task(frames):
    # frames[0] = train : c'est le seul sur lequel le prétraitement est ajusté.
    frames=[f.copy().reset_index(drop=True) for f in frames]
    good=[c for c in frames[0].columns
          if frames[0][c].notna().sum()>0 and frames[0][c].nunique(dropna=True)>1]
    if not good: raise ValueError('Aucune variable exploitable')
    nums=[c for c in good if pd.api.types.is_numeric_dtype(frames[0][c])]
    cats=[c for c in good if c not in nums]
    parts=[]
    if nums: parts.append(('numeric',SimpleImputer(strategy='median'),nums))
    if cats: parts.append(('categorical',OrdinalEncoder(handle_unknown='use_encoded_value',
                            unknown_value=-1,encoded_missing_value=-1,dtype=np.float32),cats))
    prep=ColumnTransformer(parts,remainder='drop',sparse_threshold=0)
    clean=[]
    for f in frames:
        f=f[good].copy()
        for c in nums: f[c]=pd.to_numeric(f[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        for c in cats: f[c]=f[c].astype('string').fillna('__MISSING__').astype(str)
        clean.append(f)
    out=[]
    for j,f in enumerate(clean):
        x=prep.fit_transform(f) if j==0 else prep.transform(f)
        x=np.asarray(x,dtype=np.float32)
        x=np.clip(np.nan_to_num(x,nan=0,posinf=1e4,neginf=-1e4),-1e4,1e4)
        if not np.isfinite(x).all(): raise ValueError('Valeurs X non finies')
        out.append(np.ascontiguousarray(x))
    return out,{'n_features':len(good),'numeric':len(nums),'categorical':len(cats),
                'feature_name_sha256':hashlib.sha256(json.dumps([str(c) for c in good]).encode()).hexdigest()}

def get_bundle(tid):
    task=TASKS[tid]; ds=openml_retry(task.get_dataset)
    X,y,_,_=openml_retry(ds.get_data,target=task.target_name)
    if not isinstance(X,pd.DataFrame) or isinstance(y,pd.DataFrame):
        raise TypeError('Format X/y OpenML inattendu')
    outer,sealed=task.get_train_test_split_indices(fold=0,repeat=0,sample=0)
    outer=np.asarray(outer); sealed=np.asarray(sealed)
    assert np.intersect1d(outer,sealed).size==0 and len(outer)+len(sealed)==len(X)
    label=LabelEncoder().fit(pd.Series(y).iloc[outer].astype(str))
    if len(label.classes_)!=2: raise ValueError(f'{tid}: tâche non binaire')
    y_outer=label.transform(pd.Series(y).iloc[outer].astype(str))
    if min(np.bincount(y_outer))<20: raise ValueError(f'{tid}: classe minoritaire trop rare')
    i_fit,i_report=train_test_split(outer,test_size=0.20,stratify=y_outer,random_state=SEED)
    y_fit=label.transform(pd.Series(y).iloc[i_fit].astype(str))
    i_train,i_val=train_test_split(i_fit,test_size=0.30,stratify=y_fit,random_state=SEED+1)
    y_val_tmp=label.transform(pd.Series(y).iloc[i_val].astype(str))
    i_score,i_select=train_test_split(i_val,test_size=0.50,stratify=y_val_tmp,random_state=SEED+2)
    for a,b in [(i_train,i_score),(i_train,i_select),(i_train,i_report),
                (i_score,i_select),(i_score,i_report),(i_select,i_report)]:
        assert np.intersect1d(a,b).size==0, f'{tid}: fuite entre splits'
    enc=lambda idx: label.transform(pd.Series(y).iloc[idx].astype(str)).astype(np.int64)
    ytr,ysc,ysl,yrp=enc(i_train),enc(i_score),enc(i_select),enc(i_report)
    for nm,z in [('train',ytr),('Vscore',ysc),('Vselect',ysl),('report',yrp)]:
        if min(np.bincount(z))<4: raise ValueError(f'{tid}: classe trop rare dans {nm}')
    (Xtr,Xsc,Xsl,Xrp),feat=encode_task([X.iloc[i_train],X.iloc[i_score],X.iloc[i_select],X.iloc[i_report]])
    if len(X)>MAX_DATASET_ROWS or Xtr.shape[1]>MAX_FEATURES:
        raise ValueError(f'{tid}: hors plafonds ({len(X)} lignes, {Xtr.shape[1]} variables)')
    ctx=choose_indices(ytr,min(LOCAL_TRAIN_CAP,len(ytr)),SEED+tid)
    b=dict(Xtrain=Xtr,ytrain=ytr,Xscore=Xsc,yscore=ysc,Xselect=Xsl,yselect=ysl,
           Xreport=Xrp,yreport=yrp,Xcontext=Xtr[ctx],ycontext=ytr[ctx],
           context_idx=ctx,cat_indices=[],
           meta={**META[tid],'n_train':len(ytr),'n_score':len(ysc),'n_select':len(ysl),
                 'n_report':len(yrp),'n_context':len(ctx),'features':feat,
                 'sealed_outer_test_rows':len(sealed),
                 'split_sha256':hashlib.sha256(np.sort(i_train).astype('<i8').tobytes()).hexdigest()})
    return b

def bundle_cache_paths(tid):
    # Un run precedent, ou une sortie attachee en entree, evite de redependre d'OpenML.
    cands=[ROOT/'bundles'/f'{tid}.npz']
    inp=Path('/kaggle/input')
    if inp.exists(): cands+=sorted(inp.glob(f'**/bundles/{tid}.npz'))
    return cands

def load_bundle(tid):
    for p in bundle_cache_paths(tid):
        if not p.exists(): continue
        try:
            with np.load(p,allow_pickle=False) as a:
                b={k:a[k] for k in a.files if k!='meta_json'}
                b['meta']=json.loads(str(a['meta_json'].item()))
            b['cat_indices']=[]
            return b
        except Exception as exc:
            print(f'  cache illisible {p} ({type(exc).__name__})',flush=True)
    return None

def save_bundle(tid,b):
    d=ROOT/'bundles'; d.mkdir(parents=True,exist_ok=True)
    p=d/f'{tid}.npz'; tmp=p.with_name(p.name+'.tmp')
    arrays={k:v for k,v in b.items() if isinstance(v,np.ndarray)}
    with tmp.open('wb') as f:
        np.savez_compressed(f,meta_json=np.array(json.dumps(b['meta'],default=str)),**arrays)
    tmp.replace(p)

BUNDLES={}; rows=[]; from_cache=0
for tid in ALL_TASK_IDS:
    try:
        cached=load_bundle(tid)
        if cached is not None:
            BUNDLES[tid]=cached; from_cache+=1
        else:
            check_disk(f'chargement {tid}')
            BUNDLES[tid]=get_bundle(tid); save_bundle(tid,BUNDLES[tid])
        m=BUNDLES[tid]['meta']
        rows.append({k:m[k] for k in
            ['task_id','name','n_train','n_score','n_select','n_report','n_context']}
            |{'n_features':m['features']['n_features']})
        print(f"{m['name'][:42]:<42} | train {m['n_train']:>5} Vscore {m['n_score']:>4} "
              f"Vselect {m['n_select']:>4} rapport {m['n_report']:>5} | contexte {m['n_context']:>4} "
              f"| variables {m['features']['n_features']:>3}")
    except Exception as exc:
        record_failure('bundle',tid,META[tid]['name'],exc)
        clean_gpu()

print(f'{from_cache}/{len(ALL_TASK_IDS)} datasets relus depuis un cache local ou attache.')
ACTIVE_IDS=[t for t in ALL_TASK_IDS if t in BUNDLES]
if len(ACTIVE_IDS)<MIN_TARGETS:
    raise RuntimeError(f'Seulement {len(ACTIVE_IDS)} datasets chargés : voir failures.csv.')
PARTITIONS=pd.DataFrame(rows); write_csv(ROOT/'partitions.csv',PARTITIONS); display(PARTITIONS)
write_json(ROOT/'bundles_meta.json',{str(t):BUNDLES[t]['meta'] for t in ACTIVE_IDS})
print(f'\n{len(ACTIVE_IDS)}/24 datasets actifs — chacun sert de cible et de donneur.')

## 4. Moteur épisodiqueRepris du pilote, **inchangé sauf un point** : la taille de l'épisode est devenue un paramètre, puisque c'est l'axe de la phase C. Pas de LoRA, pas de projection, pas de sous-échantillonnage des paramètres.Le repli numérique est celui qui a fait échouer ton run précédent sur `NATICUSdroid` : la T4 est en `sm_75`, elle n'a pas de bf16, l'autocast tombe donc en fp16 et le backward déborde sur les tables larges. On réduit l'échelle de perte sur 8 paliers, puis on repasse en fp32, puis on redessine l'épisode. **Le drapeau `used_amp` compte** : en fp32 les gradients sont déjà à l'échelle 1, l'appelant doit faire `optimizer.step()` et surtout pas `scaler.step()`, qui les diviserait par 65536.

In [ ]:

def stratified_pick(y,n,seed):
    y=np.asarray(y); assert len(np.unique(y))==2
    n=min(int(n),len(y))
    if n==len(y): return np.arange(len(y))
    idx,_=train_test_split(np.arange(len(y)),train_size=n,stratify=y,random_state=int(seed))
    if len(np.unique(y[idx]))<2: raise ValueError('Épisode mono-classe')
    return np.sort(idx)

def ft_episode(b,seed,n_context,n_query):
    # Contexte et requêtes disjoints au niveau des lignes, tous deux dans le train.
    for k in range(30):
        s=int(seed)+k*100003
        iq=stratified_pick(b['ytrain'],n_query,s)
        pool=np.flatnonzero(~np.isin(np.arange(len(b['ytrain'])),iq))
        if len(pool)<n_context or len(np.unique(b['ytrain'][pool]))<2: continue
        ic=pool[stratified_pick(b['ytrain'][pool],n_context,s+1)]
        assert not (set(ic.tolist()) & set(iq.tolist()))
        return (b['Xtrain'][ic],b['ytrain'][ic],b['Xtrain'][iq],b['ytrain'][iq],s),ic,iq
    raise RuntimeError('Impossible de construire un épisode disjoint')

class EpisodeEngine:
    def __init__(self,checkpoint,cat_indices,seed):
        set_seed(seed)
        kw=dict(device=DEVICE,n_estimators=1,random_state=int(seed),fit_mode='batched',
                differentiable_input=False,memory_saving_mode=True,
                categorical_features_indices=cat_indices,
                inference_config={'ENABLE_GPU_PREPROCESSING':False,'TRANSFORM_TEXT':False,
                                  'TRANSFORM_DATES':False})
        self.est=(TabPFNClassifier.create_default_for_version(ModelVersion.V3_5,**kw)
                  if checkpoint is None else TabPFNClassifier(model_path=str(checkpoint),**kw))
        self.est._initialize_model_variables(); self.est.model_.to(DEVICE)
        for p in _parameters_unused_by_task(self.est.model_,'multiclass'): p.requires_grad=False
        self.wrapper=FinetunedTabPFNClassifier(device=DEVICE,model_version=ModelVersion.V3_5)
        self.wrapper.finetuned_estimator_=self.est
        self.wrapper._ddp_module_=None; self.wrapper._local_n_estimators_=1
        self.performance=PerformanceOptions(force_recompute_layer=True,use_chunkwise_inference=False)
    def params(self):
        return {n:p for n,p in self.est.model_.named_parameters() if p.requires_grad}
    def batch(self,episode):
        xc,yc,xq,yq,seed=episode; k=len(yc)
        assert set(np.unique(yc))=={0,1}
        def fixed_split(X,y,stratify=None): return X[:k],X[k:],y[:k],y[k:]
        ds=get_preprocessed_dataset_chunks(calling_instance=self.est,
            X_raw=np.ascontiguousarray(np.concatenate([xc,xq]),dtype=np.float32),
            y_raw=np.concatenate([yc,yq]),split_fn=fixed_split,max_data_size=None,
            model_type='classifier',equal_split_size=False,data_shuffle_seed=seed,
            preprocessing_random_state=seed,shuffle=False)
        assert len(ds)==1
        return meta_dataset_collator([ds[0]])
    def loss(self,batch):
        self.est.fit_from_preprocessed(batch.X_context,batch.y_context,batch.cat_indices,
                                       batch.configs,performance_options=self.performance)
        self.est.model_.train()
        return self.wrapper._forward_with_loss(batch)
    def save(self,path):
        p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp'); save_tabpfn_model(self.est,tmp); tmp.replace(p)
    def close(self):
        del self.wrapper; del self.est; clean_gpu()

AMP_EVENTS=[]   # journal des replis numériques : une donnée de l'article

def _grads_finite(engine):
    return all(p.grad is None or bool(torch.isfinite(p.grad).all())
               for p in engine.params().values())

def backward_checked(engine,batch,optimizer,scaler,tag='',step=None):
    # Renvoie (loss, used_amp). used_amp=False => gradients déjà à l'échelle 1 :
    # l'appelant doit faire optimizer.step(), surtout pas scaler.step().
    scale0=scaler.get_scale()
    for k in range(8):                       # 65536 -> 4, par quarts successifs
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=True),sdpa_kernel_context(): loss=engine.loss(batch)
        if not torch.isfinite(loss): break   # le problème est en avant, pas en arrière
        with sdpa_kernel_context(): scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if _grads_finite(engine):
            if k: AMP_EVENTS.append(dict(tag=tag,step=step,mode='amp_rescaled',
                                         reductions=k,scale=scaler.get_scale()))
            return float(loss.detach()),True
        scaler.update(new_scale=max(1.0,scaler.get_scale()/4))
    # Repli fp32 : l'overflow fp16 disparaît, le pas reste exact.
    optimizer.zero_grad(set_to_none=True)
    with autocast(enabled=False),sdpa_kernel_context(): loss=engine.loss(batch)
    if torch.isfinite(loss):
        with sdpa_kernel_context(): loss.backward()
        if _grads_finite(engine):
            scaler.update(new_scale=scale0)
            AMP_EVENTS.append(dict(tag=tag,step=step,mode='fp32'))
            return float(loss.detach()),False
    raise FloatingPointError('Gradients non finis en AMP comme en fp32')

def named_hash(engine):
    h=hashlib.sha256()
    for name,p in sorted(engine.params().items()):
        a=p.detach().cpu().contiguous()
        h.update(name.encode()); h.update(str(tuple(a.shape)).encode())
        h.update(a.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

print('Moteur épisodique prêt.')

## 5. Entraînement jalonné, et la courbe de contexteInchangés, à une colonne près : chaque point enregistre désormais la taille d'épisodequi l'a produit (`ep_context`, `ep_query`), puisque c'est devenu la variable d'intérêt.C'est aussi ce qui permet de calculer le rapport contexte ÷ épisode, la quantité quidevrait faire s'effondrer les trois courbes sur une seule.

In [ ]:

BASE=SCRATCH/'base_weights.pth'
if not BASE.exists():
    check_disk('poids de base',extra=2.0)
    e=EpisodeEngine(None,[],SEED)
    try: e.save(BASE)
    finally: e.close()
print('Poids officiels :',BASE,'|',round(BASE.stat().st_size/GIB,2),'Gio')

def predict_ctx(checkpoint,Xctx,yctx,Xq,seed,batch,cat_indices):
    m=TabPFNClassifier(model_path=str(checkpoint),device=DEVICE,n_estimators=1,
        fit_mode='low_memory',memory_saving_mode=True,random_state=int(seed),
        categorical_features_indices=cat_indices,ignore_pretraining_limits=True)
    try:
        m.fit(Xctx,yctx); pos=list(m.classes_).index(1); parts=[]
        for st in range(0,len(Xq),batch):
            parts.append(np.asarray(m.predict_proba(Xq[st:st+batch])[:,pos],dtype=float))
        p=np.concatenate(parts)
        assert len(p)==len(Xq) and np.isfinite(p).all()
        return p
    finally:
        del m; clean_gpu()

def ctx_indices(b,size,seed):
    n=len(b['ytrain']); k=n if size is None else min(int(size),n)
    return choose_indices(b['ytrain'],k,seed)

def eval_slice(b,cap,seed):
    idx=choose_indices(b['yreport'],min(cap,len(b['yreport'])),seed)
    return b['Xreport'][idx],b['yreport'][idx]

def curve_for(checkpoint,tid,arm,ft_steps,ep_context,ep_query,
              lr=LEARNING_RATE,l2sp=0.0,regime='standard'):
    # Un jeu de poids -> une courbe complète sur l'axe du contexte.
    b=BUNDLES[tid]; cat=b['cat_indices']; ntr=len(b['ytrain'])
    Xq,yq=eval_slice(b,REPORT_EVAL_CAP,SEED+900+tid)
    out=[]
    for size in CONTEXT_SIZES:
        if size is not None and size>=ntr:
            continue                      # identique au contexte complet : on ne paie pas deux fois
        nrep=REPEATS_SMALL if (size is not None and size<=SMALL_CONTEXT_MAX) else 1
        for r in range(nrep):
            try:
                idx=ctx_indices(b,size,SEED+tid+7919*r)
            except ValueError as exc:
                print(f'  {arm} ctx={size} : tirage impossible ({exc})',flush=True); break
            t0=time.monotonic(); p=None
            for bs in (PRED_BATCH,16,4,1):
                try:
                    p=predict_ctx(checkpoint,b['Xtrain'][idx],b['ytrain'][idx],Xq,SEED,bs,cat)
                    break
                except RuntimeError as exc:
                    if not is_oom(exc): raise
                    clean_gpu()
            if p is None:
                print(f'  {arm} ctx={size} : OOM à {len(idx)} lignes, point omis',flush=True); break
            out.append(dict(run_hash=RUN_HASH,target=tid,name=META[tid]['name'],n_train=ntr,
                            arm=arm,ft_steps=int(ft_steps),
                            ep_context=int(ep_context),ep_query=int(ep_query),
                            episode_rows=int(ep_context+ep_query),
                            lr=float(lr),l2sp=float(l2sp),regime=str(regime),
                            context_size=('full' if size is None else str(int(size))),
                            requested=(ntr if size is None else int(size)),
                            n_context_rows=len(idx),repeat=r,n_report_eval=len(yq),
                            seconds=round(time.monotonic()-t0,1),source='mesuré ici',
                            **scores(yq,p)))
    if not out: raise RuntimeError(f'{arm} : aucune taille de contexte évaluable')
    return out

def displacement(engine,theta0):
    # ||theta - theta_0||, en fp32 sur CPU : c'est la seule échelle qui permette de
    # comparer deux taux d'apprentissage autrement qu'à nombre de pas égal.
    s=0.0
    for n,p in engine.params().items():
        if n in theta0: s+=float(((p.detach().float().cpu()-theta0[n])**2).sum())
    return float(np.sqrt(s))

def train_fixed(b,start,path,milestones,seed,tag,n_context,n_query,
                lr=None,wd=None,l2sp=0.0):
    lr=LEARNING_RATE if lr is None else float(lr)
    wd=WEIGHT_DECAY  if wd is None else float(wd)
    path=Path(path); path.mkdir(parents=True,exist_ok=True)
    done=path/'complete.json'; end=max(milestones)
    signature=sha(dict(run=RUN_HASH,start=sha_file(start),steps=sorted(milestones),
                       seed=seed,tag=tag,ep=(n_context,n_query),lr=lr,wd=wd,l2sp=l2sp))
    if done.exists():
        info=json.loads(done.read_text())
        if info['signature']==signature and all(
                Path(c['path']).is_file() and sha_file(c['path'])==c['sha256']
                for c in info['checkpoints'].values()):
            print(tag,': déjà entraîné, réutilisé.'); return info
    check_disk(tag,extra=2.0*len(milestones))
    e=EpisodeEngine(start,b['cat_indices'],seed)
    opt=torch.optim.AdamW(list(e.params().values()),lr=lr,weight_decay=wd)
    scaler=GradScaler(); history=[]; ckpts={}; seen=set(); redraws=0
    theta0={n:p.detach().float().cpu().clone() for n,p in e.params().items()}
    anchor={n:t.to(DEVICE) for n,t in theta0.items()} if l2sp>0 else None
    t0=time.monotonic(); torch.cuda.reset_peak_memory_stats(); initial=named_hash(e)
    try:
        for step in range(1,end+1):
            for attempt in range(6):
                episode,ic,iq=ft_episode(b,seed+step*7919+attempt*104729,n_context,n_query)
                batch=e.batch(episode)
                try:
                    loss,used_amp=backward_checked(e,batch,opt,scaler,tag=tag,step=step)
                    break
                except FloatingPointError:
                    redraws+=1
                    AMP_EVENTS.append(dict(tag=tag,step=step,mode='episode_redrawn',attempt=attempt))
                    del batch; clean_gpu()
            else:
                raise FloatingPointError(
                    f'{tag} : 6 épisodes consécutifs non finis au pas {step}. '
                    'Instabilité reproductible, pas un aléa : signaler ce dataset.')
            seen.update(ic.tolist()); seen.update(iq.tolist())
            if anchor is not None:
                # L2-SP : penalite alpha/2 * ||theta - theta_0||^2, ajoutee au gradient
                # avant le clipping, comme une penalite de perte et non un decay decouple.
                for n,p in e.params().items():
                    if p.grad is not None and n in anchor:
                        p.grad.add_(p.detach()-anchor[n],alpha=l2sp)
            norm=float(torch.nn.utils.clip_grad_norm_(list(e.params().values()),GRAD_CLIP))
            if used_amp:
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            history.append(dict(step=step,loss=loss,grad_norm_before_clip=norm,
                                amp=used_amp,seconds=time.monotonic()-t0))
            del batch
            if step in milestones:
                ck=path/f'step_{step:04d}.pth'; e.save(ck)
                wh=named_hash(e); dsp=displacement(e,theta0)
                assert wh!=initial,'Poids inchangés après entraînement'
                ckpts[str(step)]=dict(path=str(ck),sha256=sha_file(ck),weights_hash=wh,
                                      displacement=dsp,seconds=time.monotonic()-t0)
                write_csv(path/'training_history.csv',pd.DataFrame(history))
                fp32=int((~pd.DataFrame(history).amp).sum())
                print(f'  {tag} | pas {step}/{end} | loss {loss:.5f} | |g| {norm:.3f} '
                      f'| déplacement {dsp:.4f}'
                      +(f' | replis fp32 {fp32} | redraws {redraws}' if fp32 or redraws else ''),
                      flush=True)
        info=dict(signature=signature,initial_hash=initial,checkpoints=ckpts,
                  seconds=time.monotonic()-t0,n_train=len(b['ytrain']),lr=lr,wd=wd,l2sp=l2sp,
                  unique_rows_seen=len(seen),episode_rows=n_context+n_query,
                  seconds_per_step=(time.monotonic()-t0)/end,
                  fp32_steps=int((~pd.DataFrame(history).amp).sum()),
                  episode_redraws=redraws,
                  final_loss_mean=float(pd.DataFrame(history).loss.tail(10).mean()),
                  peak_allocated_gib=torch.cuda.max_memory_allocated()/GIB)
        write_json(done,info)
        return info
    finally:
        del opt,scaler,theta0,anchor; e.close()
        if AMP_EVENTS: write_csv(ROOT/'amp_events.csv',pd.DataFrame(AMP_EVENTS))

print('Entraînement jalonné et courbe de contexte prêts.')

## 6. Prétest mémoire et protocoleIdentique au run précédent : quel lot de requêtes tient, le contexte complet du plus grosdataset passe-t-il, les poids bougent-ils, le rechargement est-il exact, et un modèleadapté prédit-il bien autrement que le modèle de base.

In [ ]:

attempts=[]; PRED_BATCH=None
for bs in PRED_BATCHES:
    try:
        for tid in ACTIVE_IDS:
            b=BUNDLES[tid]
            predict_ctx(BASE,b['Xcontext'],b['ycontext'],b['Xreport'][:bs],SEED,bs,b['cat_indices'])
        PRED_BATCH=bs; break
    except RuntimeError as exc:
        if not is_oom(exc): raise
        attempts.append(dict(batch=bs,error='CUDA OOM')); clean_gpu()
if PRED_BATCH is None: raise RuntimeError('Même une requête unique ne tient pas en mémoire.')
print('Lot de requêtes retenu :',PRED_BATCH)

from sklearn.datasets import load_breast_cancer
xp,yp=load_breast_cancer(return_X_y=True)
a_,v_=train_test_split(np.arange(len(yp)),train_size=400,stratify=yp,random_state=SEED)
probe=dict(Xtrain=xp[a_].astype('float32'),ytrain=yp[a_].astype(np.int64),
           Xreport=xp[v_].astype('float32'),yreport=yp[v_].astype(np.int64),
           Xcontext=xp[a_].astype('float32'),ycontext=yp[a_].astype(np.int64),cat_indices=[])
RUN_HASH='preflight'
pf=SCRATCH/'_preflight'
if pf.exists(): shutil.rmtree(pf)
try:
    info=train_fixed(probe,BASE,pf,[1,2],SEED,'prétest',102,26)
    ck=info['checkpoints']['2']['path']
    e=EpisodeEngine(ck,[],SEED)
    try: assert named_hash(e)==info['checkpoints']['2']['weights_hash'],'Rechargement incorrect'
    finally: e.close()
    q=probe['Xreport'][:16]
    p1=predict_ctx(ck,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    p0=predict_ctx(BASE,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    assert float(np.abs(p0-p1).max())>1e-4,'Modèle adapté identique au modèle de base'
    print('PRÉTEST PASS — écart base/adapté max',round(float(np.abs(p0-p1).max()),5))
finally:
    shutil.rmtree(pf,ignore_errors=True); clean_gpu()

TASK_IDS=[t for t in ACTIVE_IDS if t in SHARD_IDS]

def eligible_for(nc,nq):
    need=int(ELIG_SLACK*(nc+nq))
    return [t for t in TASK_IDS if len(BUNDLES[t]['ytrain'])>=need]

PROTOCOL=dict(seed=SEED,splits='outer_80_20_then_70_15_15_v1',
    context_sizes=[str(c) for c in CONTEXT_SIZES],ref_context=REF_CONTEXT,
    repeats_small=REPEATS_SMALL,small_max=SMALL_CONTEXT_MAX,report_cap=REPORT_EVAL_CAP,
    lr=LEARNING_RATE,wd=WEIGHT_DECAY,clip=GRAD_CLIP)
RUN_HASH=sha(dict(protocol=PROTOCOL,tasks=TASK_IDS,milestones=list(FT_MILESTONES),
                  episodes=[list(e) for e in EPISODE_FIXED]))[:16]
RUN_T0=time.monotonic()
def hours_left(): return TIME_BUDGET_HOURS-(time.monotonic()-RUN_T0)/3600
print('Protocole :',RUN_HASH,'|',len(TASK_IDS),'datasets')
for nc,nq in EPISODE_FIXED:
    print(f'  épisodes de {nc+nq:>4} lignes : {len(eligible_for(nc,nq)):>2} datasets éligibles')

## 7. Import du run précédentOn ne fait confiance à rien : l'import ne se contente pas de lire le CSV, il **vérifie**.1. Le `manifest.json` attaché doit déclarer exactement le même protocole — même graine,   mêmes splits, mêmes tailles de contexte, mêmes hyperparamètres de fine-tuning. Sinon   la source est ignorée, et on dit laquelle.2. Les bras importés sont renommés d'après la taille d'épisode déclarée dans *son*   manifeste, jamais d'après une convention supposée.3. Puis on **re-mesure** la courbe du modèle de base sur trois datasets et on la compare   à la version importée. C'est le contrôle qui compte : il teste d'un coup les splits,   le tirage des contextes, la tranche d'évaluation et les poids de départ. Si l'écart   dépasse la tolérance, **tout l'import est rejeté** et le run recalcule — on ne   rafistole pas un jeu de données mixte.Ce que ce contrôle ne peut pas couvrir : les bras entraînés, qu'on ne peut revérifiersans les réentraîner. Si la courbe de base se reproduit au millième près, le harnais estle même, et c'est le meilleur argument disponible.

In [ ]:

CURVE_PATH=ROOT/'curves.csv'
curves=pd.read_csv(CURVE_PATH).to_dict('records') if CURVE_PATH.exists() else []

def collect_previous():
    found=[]
    root=Path('/kaggle/input')
    if not (IMPORT_PREVIOUS and root.exists()): return found
    for man in sorted(root.glob('**/manifest.json')):
        csv=man.parent/'curves.csv'
        if not csv.exists(): continue
        try: cfg=json.loads(man.read_text()).get('config',{})
        except Exception as exc:
            print(f'  {man.parent.name} : manifeste illisible ({type(exc).__name__})'); continue
        bad=[k for k,v in PROTOCOL.items() if cfg.get(k)!=v]
        if bad:
            print(f'  {man.parent.name} : protocole différent sur {bad} — ignoré',flush=True); continue
        mapping={'base':('base',0,0,0)}
        ep=tuple(cfg.get('ft_episode') or ()); lp=tuple(cfg.get('long_episode') or ())
        if len(ep)==2:
            for s in cfg.get('ft_milestones',[]):
                mapping[f'ft_{s}']=(arm_name(sum(ep),s),int(s),int(ep[0]),int(ep[1]))
        if len(lp)==2:
            for s in cfg.get('long_milestones',[]):
                mapping[f'ftlong_{s}']=(arm_name(sum(lp),s),int(s),int(lp[0]),int(lp[1]))
        df=pd.read_csv(csv)
        df=df[df.arm.isin(mapping)].copy()
        if df.empty:
            print(f'  {man.parent.name} : aucun bras reconnu — ignoré'); continue
        df['ft_steps']=[mapping[a][1] for a in df.arm]
        df['ep_context']=[mapping[a][2] for a in df.arm]
        df['ep_query']=[mapping[a][3] for a in df.arm]
        df['episode_rows']=df.ep_context+df.ep_query
        # Le run importé n'avait qu'un régime d'optimisation ; on l'étiquette comme tel.
        df['lr']=float(cfg.get('lr',LEARNING_RATE)); df['l2sp']=0.0; df['regime']='standard'
        df['source']=man.parent.name
        df['arm']=[mapping[a][0] for a in df.arm]
        df['run_hash']=RUN_HASH
        found.append(df)
        print(f'  {man.parent.name} : {len(df)} points, bras {sorted(df.arm.unique())}',flush=True)
    return found

def verify_import(df,n_check):
    # Re-mesure la courbe de base sur quelques datasets et compare à l'importé.
    got=df[df.arm=='base']
    tids=[t for t in TASK_IDS if t in set(got.target)][:n_check]
    worst=0.0; rows=[]
    for tid in tids:
        fresh=pd.DataFrame(curve_for(BASE,tid,'base',0,0,0))
        a=(fresh.groupby('context_size').logloss.mean())
        b_=(got[got.target==tid].groupby('context_size').logloss.mean())
        common=a.index.intersection(b_.index)
        d=float((a[common]-b_[common]).abs().max()) if len(common) else np.inf
        worst=max(worst,d)
        rows.append(dict(target=tid,name=META[tid]['name'],points=len(common),ecart_max=round(d,6)))
        print(f'  {META[tid]["name"][:34]:<34} écart max de log-loss : {d:.2e}',flush=True)
    return worst,pd.DataFrame(rows),tids

if not curves:
    print('=== IMPORT DU RUN PRÉCÉDENT ===',flush=True)
    found=collect_previous()
    if not found:
        print('Aucune sortie réutilisable attachée : tout sera recalculé.',flush=True)
    else:
        IMP=pd.concat(found,ignore_index=True).drop_duplicates(
            subset=['target','arm','context_size','repeat'],keep='first')
        print(f'\n{len(IMP)} points candidats sur {IMP.target.nunique()} datasets. Vérification...',flush=True)
        worst,CHECK,checked=verify_import(IMP,IMPORT_CHECK_N)
        write_csv(ROOT/'import_check.csv',CHECK)
        if worst<=IMPORT_TOLERANCE:
            print(f'IMPORT ACCEPTÉ — écart max {worst:.2e} <= {IMPORT_TOLERANCE}')
            IMP=IMP[~((IMP.arm=='base')&(IMP.target.isin(checked)))]   # on garde les mesures fraîches
            curves=pd.concat([IMP,pd.DataFrame(
                [r for t in checked for r in curve_for(BASE,t,'base',0,0,0)])],
                ignore_index=True).to_dict('records')
            write_csv(CURVE_PATH,pd.DataFrame(curves))
        else:
            print(f'IMPORT REJETÉ — écart max {worst:.2e} > {IMPORT_TOLERANCE}. '
                  'Le protocole diffère malgré un manifeste compatible : tout est recalculé.',flush=True)
            curves=[]
else:
    print(f'{len(curves)} points déjà présents dans {CURVE_PATH.name} : reprise.',flush=True)

def have(tid,arm):
    return any(int(r['target'])==tid and r['arm']==arm for r in curves)
CUR0=pd.DataFrame(curves)
if len(CUR0):
    print('\nÉtat de la grille :')
    print(CUR0.groupby('arm').target.nunique().to_string())
print(f'\nBudget restant : {hours_left():.2f} h')

## 8. Prétest : la troisième taille d'épisodeElle n'est pas choisie d'avance. Le prétest essaie les candidats **du plus grand au pluspetit** et retient le premier qui satisfait trois conditions :1. un pas d'entraînement complet passe en mémoire sur la T4, sur le dataset le plus large   du catalogue — c'est celui qui coûte le plus cher, et c'est celui qui avait fait   tomber `NATICUSdroid` au run des donneurs ;2. au moins dix datasets ont un train assez grand pour fournir de tels épisodes ;3. le coût estimé tient dans le budget de temps restant — l'estimation utilise le temps   par pas **mesuré** pendant le test, pas une extrapolation théorique.Si aucun candidat ne tient dans le budget, le prétest réduit le nombre de datasets, engardant les plus grands, plutôt que de rétrécir les épisodes : c'est la taille d'épisodequi est la variable de l'expérience, pas la taille de l'échantillon.L'arithmétique est imprimée. Si elle te déplaît, change `EPISODE_CANDIDATES` et relance.

In [ ]:

def time_one_step(nc,nq,tid):
    # Un vrai pas d'entraînement, chronométré, puis tout est libéré.
    b=BUNDLES[tid]; e=EpisodeEngine(BASE,b['cat_indices'],SEED)
    opt=torch.optim.AdamW(list(e.params().values()),lr=LEARNING_RATE)
    scaler=GradScaler()
    try:
        episode,_,_=ft_episode(b,SEED,nc,nq)
        t0=time.monotonic()
        batch=e.batch(episode)
        loss,used=backward_checked(e,batch,opt,scaler,tag='probe',step=0)
        if used: scaler.step(opt); scaler.update()
        else: opt.step()
        torch.cuda.synchronize()
        dt=time.monotonic()-t0
        del batch
        return dt,torch.cuda.max_memory_allocated()/GIB
    finally:
        del opt,scaler; e.close()

CURVE_SECONDS=CURVE_SECONDS_GUESS
if len(CUR0) and 'seconds' in CUR0.columns:
    per=CUR0[CUR0.source=='mesuré ici'].groupby(['target','arm']).seconds.sum()
    if len(per): CURVE_SECONDS=max(30.0,float(per.mean()))   # un 0 fausserait tout le budget
print(f'Coût d une courbe complète : {CURVE_SECONDS:.0f} s (mesuré ou estimé).')

EP_LARGE=None; EP_LARGE_IDS=[]; probe_log=[]
for nc,nq in EPISODE_CANDIDATES:
    rows_=nc+nq; elig=eligible_for(nc,nq)
    if len(elig)<MIN_ELIGIBLE:
        probe_log.append(dict(episode=rows_,verdict=f'{len(elig)} datasets éligibles < {MIN_ELIGIBLE}'))
        print(f'  {rows_:>4} lignes : seulement {len(elig)} datasets éligibles — écarté'); continue
    wide=max(elig,key=lambda t:BUNDLES[t]['Xtrain'].shape[1])
    try:
        dt,peak=time_one_step(nc,nq,wide)
    except RuntimeError as exc:
        if not is_oom(exc): raise
        clean_gpu()
        probe_log.append(dict(episode=rows_,verdict='OOM'))
        print(f'  {rows_:>4} lignes : OOM sur {META[wide]["name"]} — écarté'); continue
    except FloatingPointError:
        clean_gpu()
        probe_log.append(dict(episode=rows_,verdict='gradients non finis'))
        print(f'  {rows_:>4} lignes : gradients non finis — écarté'); continue
    finally:
        clean_gpu()
    per_ds=max(FT_MILESTONES)*dt+len(FT_MILESTONES)*CURVE_SECONDS
    budget=max(0.0,hours_left()-0.4)*3600
    kmax=int(budget//per_ds) if per_ds>0 else 0
    print(f'  {rows_:>4} lignes : {dt:.2f} s/pas, pic {peak:.1f} Gio, {len(elig)} éligibles, '
          f'{per_ds/60:.1f} min/dataset -> {kmax} datasets tiennent dans {budget/3600:.2f} h',flush=True)
    probe_log.append(dict(episode=rows_,seconds_per_step=round(dt,3),peak_gib=round(peak,2),
                          eligible=len(elig),minutes_per_dataset=round(per_ds/60,1),fits=kmax))
    if kmax>=MIN_ELIGIBLE:
        EP_LARGE=(nc,nq)
        EP_LARGE_IDS=sorted(elig,key=lambda t:-len(BUNDLES[t]['ytrain']))[:min(kmax,len(elig))]
        break
write_csv(ROOT/'probe_episode.csv',pd.DataFrame(probe_log))

EPISODE_GRID=[(nc,nq,eligible_for(nc,nq)) for nc,nq in EPISODE_FIXED]
if EP_LARGE is not None:
    EPISODE_GRID.append((EP_LARGE[0],EP_LARGE[1],EP_LARGE_IDS))
    print(f'\n3e taille retenue : {sum(EP_LARGE)} lignes ({EP_LARGE[0]} de contexte) '
          f'sur {len(EP_LARGE_IDS)} datasets.')
else:
    print('\nAucune 3e taille ne tient : le run se limitera à compléter 128 et 640 lignes. '
          'C est déjà ce qui transforme le n=4 en n=24.')
write_json(ROOT/'manifest.json',dict(run_hash=RUN_HASH,protocol=PROTOCOL,tasks=TASK_IDS,
    milestones=list(FT_MILESTONES),
    episodes=[[nc,nq,len(ids)] for nc,nq,ids in EPISODE_GRID],
    memory_attempts=attempts,curve_seconds=CURVE_SECONDS))

## 9. Prédiction enregistréeImprimée avant tout calcul de résultat. Les deux hypothèses font des prédictionsdifférentes **hors de la diagonale** ; c'est là et nulle part ailleurs qu'elles sedépartagent.

In [ ]:

PREREG = '''
A1  EFFONDREMENT. Le gain ne dépend pas séparément du contexte et de l épisode, mais de
    leur rapport r = contexte / contexte-d-episode. Test : Spearman(gain, log2 r) au sein
    de chaque dataset, toutes tailles d episode confondues, comparé à Spearman(gain,
    log2 contexte). Prédiction : le rapport explique MIEUX, et les trois courbes tracées
    contre r se superposent.

A2  ASYMÉTRIE. Le gain est >= 0 pour r <= 1 (on déploie plus court qu on a entraîné) et
    nettement < 0 pour r >= 2. Test : Wilcoxon par tranche de r.

A3  RÉFUTATION DE LA CORRESPONDANCE STRICTE. À contexte c fixé, l épisode PLUS GRAND que
    c n est pas moins bon que l épisode le plus proche de c. Test : comparaison appariée
    sur les datasets éligibles aux deux tailles. Si l épisode apparié gagne, c est
    l hypothèse de correspondance qui est vraie et il faut le dire.

A4  DÉPLACEMENT DU CROISEMENT. Le contexte de croisement (là où le gain change de signe)
    croît avec la taille de l épisode. Test : Spearman par dataset, puis test des signes.

A5  ROBUSTESSE À L OPTIMISATION. Avec un taux d apprentissage 10x plus petit, un decay
    nul et un ancrage L2-SP (le régime de Real-TabPFN), la FORME de la courbe est
    inchangée une fois les deux régimes comparés à DÉPLACEMENT DE POIDS égal et non à
    nombre de pas égal. Prédiction : le dommage aux grands contextes persiste. S il
    disparaît, il y a deux mécanismes et non un, et il faut le dire.

ISSUE ALTERNATIVE, à écrire telle quelle si elle se produit :
    les courbes ne se superposent pas contre r, et l épisode apparié bat l épisode plus
    grand. L asymétrie serait alors fausse, la correspondance vraie, et la recette
    pratique bien plus coûteuse : un fine-tuning par taille de contexte.
'''
print(PREREG)
write_json(ROOT/'preregistration.json',dict(run_hash=RUN_HASH,text=PREREG,
    written_before_any_result=True))

## 10. La grilleUne taille d'épisode à la fois, un dataset à la fois. Ce qui est déjà dans `curves.csv`— importé ou calculé dans une session précédente — n'est jamais recalculé. Les poids sonteffacés dès le dataset évalué : trois jalons pèsent près de 3 Gio.La garde de temps s'exerce entre deux datasets, jamais au milieu d'un entraînement.

In [ ]:

FT_DIR=SCRATCH/'ft'; FT_DIR.mkdir(parents=True,exist_ok=True)
TRAIN_PATH=ROOT/'training_summary.csv'
train_rows=pd.read_csv(TRAIN_PATH).to_dict('records') if TRAIN_PATH.exists() else []

def log_training(tid,rows_,nc,nq,info,regime):
    global train_rows
    train_rows=[r for r in train_rows
                if not (int(r['target'])==tid and int(r['episode_rows'])==rows_
                        and r.get('regime',"standard")==regime)]
    for s,c in info['checkpoints'].items():
        train_rows.append(dict(target=tid,name=META[tid]['name'],episode_rows=rows_,
            ep_context=nc,ep_query=nq,regime=regime,lr=info['lr'],l2sp=info['l2sp'],
            steps=int(s),displacement=c['displacement'],seconds=c['seconds'],
            seconds_per_step=info['seconds_per_step'],fp32_steps=info['fp32_steps'],
            episode_redraws=info['episode_redraws'],unique_rows_seen=info['unique_rows_seen'],
            final_loss_mean=info['final_loss_mean'],peak_gib=info['peak_allocated_gib']))
    write_csv(TRAIN_PATH,pd.DataFrame(train_rows))

BASE_IDS=[t for t in TASK_IDS if not have(t,'base')]
if BASE_IDS:
    print(f'=== Courbe de base manquante sur {len(BASE_IDS)} datasets ===',flush=True)
    for k,tid in enumerate(BASE_IDS,1):
        if hours_left()<=0: print('Garde de temps atteinte.',flush=True); break
        try:
            t0=time.monotonic(); new=curve_for(BASE,tid,'base',0,0,0)
            curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
            print(f'{k}/{len(BASE_IDS)} {META[tid]["name"][:34]:<34} | {time.monotonic()-t0:.0f}s',flush=True)
        except Exception as exc:
            record_failure('base',tid,META[tid]['name'],exc); clean_gpu()

for nc,nq,ids in EPISODE_GRID:
    rows_=nc+nq
    todo=[t for t in ids if not all(have(t,arm_name(rows_,s)) for s in FT_MILESTONES)]
    print(f'\n=== ÉPISODES DE {rows_} LIGNES — {len(ids)} éligibles, {len(todo)} à faire ===',flush=True)
    for k,tid in enumerate(todo,1):
        if hours_left()<=0.15:
            print('Garde de temps atteinte : on passe à l analyse de ce qui est mesuré.',flush=True); break
        b=BUNDLES[tid]; path=FT_DIR/f'ep{rows_}_{tid}'
        print(f'\n----- {k}/{len(todo)} {META[tid]["name"]} | train {len(b["ytrain"])} '
              f'| épisode {rows_} l. | {hours_left():.2f} h restantes -----',flush=True)
        try:
            check_disk(f'fine-tuning {tid}',extra=2.0*len(FT_MILESTONES))
            info=train_fixed(b,BASE,path,FT_MILESTONES,SEED+tid,
                             f'ep{rows_} {META[tid]["name"][:20]}',nc,nq)
            log_training(tid,rows_,nc,nq,info,'standard')
            for s in FT_MILESTONES:
                arm=arm_name(rows_,s)
                if have(tid,arm): continue
                new=curve_for(info['checkpoints'][str(s)]['path'],tid,arm,s,nc,nq)
                curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
                base={r['context_size']:r['logloss'] for r in curves
                      if int(r['target'])==tid and r['arm']=='base' and int(r['repeat'])==0}
                g={r['context_size']:round(base.get(r['context_size'],np.nan)-r['logloss'],4)
                   for r in new if r['repeat']==0}
                print(f'  {arm} | gain par contexte : {g}',flush=True)
        except Exception as exc:
            record_failure(f'ep{rows_}',tid,META[tid]['name'],exc); clean_gpu()
        finally:
            shutil.rmtree(path,ignore_errors=True); clean_gpu()

CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\nGrille : {CUR.target.nunique()} datasets, {CUR.arm.nunique()} bras, {len(CUR)} points. '
      f'{len(FAILURES)} écart(s). Budget restant : {hours_left():.2f} h.')

## 10 bis. L'axe de l'optimisation — la réponse à Real-TabPFN[Real-TabPFN](https://arxiv.org/abs/2507.03971) obtient des **gains** en poursuivant lepré-entraînement de TabPFN sur des données réelles, avec un taux d'apprentissage de3·10⁻⁷, un decay nul, et une pénalité L2-SP qui ancre les poids à leur valeur d'origine.Tes résultats négatifs sont obtenus à 1·10⁻⁵ sans ancrage. Un relecteur qui connaît cepapier dira, et il aura une part de raison : *vous ne mesurez pas le fine-tuning, vousmesurez un pas trop grand.*Cette phase y répond, avec une précaution qui est le cœur du contrôle : **on ne comparepas à nombre de pas égal, on compare à déplacement de poids égal.** Deux cents pas à10⁻⁵ et mille pas à 10⁻⁶ ne sont pas le même entraînement, mais ils amènent les poids àpeu près aussi loin de leur point de départ. C'est cette distance, ‖θ − θ₀‖, qui estenregistrée à chaque jalon de **toutes** les phases — et c'est elle, pas le compteur depas, qui met les deux régimes sur le même axe.Pourquoi 10⁻⁶ et pas 3·10⁻⁷ : à 3·10⁻⁷, deux cents pas ne déplaceraient presque rien etl'expérience répondrait « aucun effet » pour une raison triviale. Il faudrait environ6 600 pas pour rejoindre le déplacement du régime standard, ce qui ne tient pas dans unesession. Dix fois plus petit avec cinq fois plus de pas est le compromis qui laissel'ancrage L2-SP s'exprimer tout en atteignant un déplacement comparable.**Ce que ça peut donner.** Si la courbe garde la même forme, ta loi est robuste àl'agressivité de l'optimisation et l'objection tombe. Si le dommage disparaît à petitpas, ton résultat devient plus riche et non plus pauvre : deux mécanismes distincts, ledésaccord de longueur **et** l'ampleur du déplacement, qu'il faudra alors démêler.

In [ ]:

nc,nq=LR_EPISODE; rows_=nc+nq
# La comparaison se fait à déplacement de poids apparié : on privilégie donc les
# datasets dont le bras standard a été ENTRAÎNÉ ici, seuls à avoir un déplacement
# mesuré. Les bras importés du run précédent n'en ont pas — ils sont antérieurs à
# cette mesure — et ne pourraient pas servir de référence.
trained_std=set()
if Path(TRAIN_PATH).exists():
    _ts=pd.read_csv(TRAIN_PATH)
    if len(_ts) and 'displacement' in _ts.columns:
        _m=_ts[(_ts.regime=='standard')&(_ts.episode_rows==rows_)&_ts.displacement.notna()]
        trained_std={int(t) for t in _m.target}
_pool=[t for t in eligible_for(nc,nq) if t in trained_std] or eligible_for(nc,nq)
LR_IDS=(sorted(_pool,key=lambda t:-len(BUNDLES[t]['ytrain']))[:LR_PHASE_TASKS]
        if RUN_LR_PHASE else [])
if RUN_LR_PHASE and not trained_std:
    print('ATTENTION : aucun bras standard entraîné ici à cette taille d épisode. '
          'La comparaison à déplacement apparié sera impossible.',flush=True)
if LR_IDS and hours_left()>0.5:
    print(f'\n=== AXE OPTIMISATION — lr={LR_LOW}, L2-SP alpha={L2SP_ALPHA}, '
          f'jalons {LR_MILESTONES}, épisodes de {rows_} lignes ===',flush=True)
    for k,tid in enumerate(LR_IDS,1):
        arms=[arm_name(rows_,s,'lrlo') for s in LR_MILESTONES]
        if all(have(tid,a) for a in arms):
            print(f'{k}/{len(LR_IDS)} {META[tid]["name"]} : déjà fait.'); continue
        if hours_left()<=0.15:
            print('Garde de temps atteinte : axe optimisation interrompu.',flush=True); break
        b=BUNDLES[tid]; path=FT_DIR/f'lrlo_{tid}'
        print(f'\n----- {k}/{len(LR_IDS)} {META[tid]["name"]} | train {len(b["ytrain"])} '
              f'| {hours_left():.2f} h restantes -----',flush=True)
        try:
            check_disk(f'lr faible {tid}',extra=2.0*len(LR_MILESTONES))
            info=train_fixed(b,BASE,path,LR_MILESTONES,SEED+tid,
                             f'lrlo {META[tid]["name"][:20]}',nc,nq,
                             lr=LR_LOW,wd=LR_LOW_WD,l2sp=L2SP_ALPHA)
            log_training(tid,rows_,nc,nq,info,'lrlo')
            for s in LR_MILESTONES:
                arm=arm_name(rows_,s,'lrlo')
                if have(tid,arm): continue
                new=curve_for(info['checkpoints'][str(s)]['path'],tid,arm,s,nc,nq,
                              lr=LR_LOW,l2sp=L2SP_ALPHA,regime='lrlo')
                curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
                base={r['context_size']:r['logloss'] for r in curves
                      if int(r['target'])==tid and r['arm']=='base' and int(r['repeat'])==0}
                g={r['context_size']:round(base.get(r['context_size'],np.nan)-r['logloss'],4)
                   for r in new if r['repeat']==0}
                print(f'  {arm} | gain par contexte : {g}',flush=True)
        except Exception as exc:
            record_failure('lrlo',tid,META[tid]['name'],exc); clean_gpu()
        finally:
            shutil.rmtree(path,ignore_errors=True); clean_gpu()
else:
    print('\nAxe optimisation non exécuté (désactivé, ou budget insuffisant).')

CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\nTotal : {CUR.target.nunique()} datasets, {CUR.arm.nunique()} bras, {len(CUR)} points. '
      f'Budget restant : {hours_left():.2f} h.')

## 11. RésultatsCinq lectures. La troisième est celle qui décide entre les deux hypothèses ; les autresla situent.

In [ ]:

from scipy.stats import wilcoxon, spearmanr, binomtest

CUR=pd.read_csv(CURVE_PATH)
for c_,v_ in [('regime','standard'),('lr',LEARNING_RATE),('l2sp',0.0)]:
    if c_ not in CUR.columns: CUR[c_]=v_
KEY=['target','name','n_train','arm','ft_steps','ep_context','ep_query','episode_rows',
     'regime','lr','l2sp','context_size','requested']
AGG=(CUR.groupby(KEY,as_index=False)
        .agg(logloss=('logloss','mean'),auc=('auc','mean'),ap=('ap','mean'),
             n_context_rows=('n_context_rows','mean'),repeats=('repeat','count')))
AGG['ctx']=AGG.n_context_rows.astype(float)
write_csv(ROOT/'curves_agg.csv',AGG)

FT=AGG[AGG.arm!='base'].merge(AGG[AGG.arm=='base'][['target','context_size','logloss','auc']],
    on=['target','context_size'],suffixes=('','_base'),how='inner')
FT['gain_logloss']=FT.logloss_base-FT.logloss
FT['gain_auc']=FT.auc-FT.auc_base
FT['ratio']=FT.ctx/FT.ep_context.clip(lower=1)
write_csv(ROOT/'ft_gains.csv',FT)
LR_PROBE=FT[FT.regime=='lrlo'].copy()
FT=FT[FT.regime=='standard'].copy()      # la grille principale est à régime constant
TOP=max(FT_MILESTONES)
SIZES=sorted(FT.episode_rows.unique())
print('Tailles d épisode mesurées :',SIZES)
print('Datasets par taille :',{int(s):int(FT[FT.episode_rows==s].target.nunique()) for s in SIZES})

print('\n=== 1. LA GRILLE, sur le SOUS-ENSEMBLE COMMUN aux tailles d épisode ===')
common=set.intersection(*[set(FT[FT.episode_rows==s].target) for s in SIZES]) if SIZES else set()
print(f'{len(common)} datasets éligibles à toutes les tailles.')
CO=FT[FT.target.isin(common)]
for st in FT_MILESTONES:
    sub=CO[CO.ft_steps==st]
    if sub.empty: continue
    t=(sub.pivot_table(index='context_size',columns='episode_rows',values='gain_logloss',
                       aggfunc='mean')
         .join(sub.groupby('context_size').ctx.mean().rename('lignes'))
         .sort_values('lignes').round(5))
    print(f'\nGain de log-loss à {st} pas — colonnes = lignes par épisode, n={len(common)}')
    display(t); write_csv(ROOT/f'grille_{st}pas.csv',t.reset_index())

print('\n=== 2. TÊTE À TÊTE entre tailles d épisode, à chaque contexte ===')
h2h=[]
for i in range(len(SIZES)):
    for j in range(i+1,len(SIZES)):
        a,b_=SIZES[i],SIZES[j]
        for cs,g in FT[FT.ft_steps==TOP].groupby('context_size'):
            p=g.pivot_table(index='target',columns='episode_rows',values='gain_logloss')
            if a not in p or b_ not in p: continue
            d=(p[b_]-p[a]).dropna()
            if len(d)<5 or d.abs().sum()==0: continue
            h2h.append(dict(contexte=cs,lignes=round(float(g.ctx.mean()),1),
                comparaison=f'{int(b_)} l. - {int(a)} l.',ecart_moyen=round(float(d.mean()),5),
                ecart_median=round(float(d.median()),5),
                wilcoxon_p=round(float(wilcoxon(d).pvalue),5),
                victoires_grand=int((d>0).sum()),n=len(d)))
H2H=pd.DataFrame(h2h)
if len(H2H):
    H2H=H2H.sort_values(['comparaison','lignes'])
    display(H2H); write_csv(ROOT/'tete_a_tete.csv',H2H)

print('\n=== 3. A1/A2 — LE RAPPORT EXPLIQUE-T-IL MIEUX QUE LE CONTEXTE ? ===')
cmp_rows=[]
for tid,g in FT[FT.ft_steps==TOP].groupby('target'):
    g=g.dropna(subset=['gain_logloss'])
    if g.episode_rows.nunique()<2 or len(g)<6: continue
    r1=spearmanr(np.log2(g.ratio),g.gain_logloss)[0]
    r2=spearmanr(np.log2(g.ctx),g.gain_logloss)[0]
    cmp_rows.append(dict(target=tid,name=g.name.iloc[0],n_train=int(g.n_train.iloc[0]),
        rho_rapport=round(float(r1),4),rho_contexte=round(float(r2),4),
        gagne_le_rapport=bool(abs(r1)>abs(r2)),points=len(g)))
CMP=pd.DataFrame(cmp_rows)
if len(CMP):
    display(CMP.sort_values('rho_rapport'))
    write_csv(ROOT/'rapport_vs_contexte.csv',CMP)
    w=int(CMP.gagne_le_rapport.sum()); n=len(CMP)
    print(f'A1 : le rapport explique mieux que le contexte absolu sur {w}/{n} datasets '
          f'(|rho| médian {CMP.rho_rapport.abs().median():.3f} contre '
          f'{CMP.rho_contexte.abs().median():.3f}, test des signes p={binomtest(w,n,0.5).pvalue:.5f}).')
    print('  -> effondrement soutenu' if w>n/2 and binomtest(w,n,0.5).pvalue<0.05 else
          '  -> effondrement NON soutenu : le dire tel quel.')

print('\nA2 — le gain par tranche de rapport (contexte / contexte-d-épisode)')
EDGES=[0,0.25,0.5,1.0,2.0,4.0,8.0,np.inf]
LABS=['<=1/4','1/4-1/2','1/2-1','1-2','2-4','4-8','>8']
tr=[]
sub=FT[FT.ft_steps==TOP].dropna(subset=['gain_logloss']).copy()
sub['tranche']=pd.cut(sub.ratio,EDGES,labels=LABS,right=True,include_lowest=True)
for lab,g in sub.groupby('tranche',observed=True):
    d=g.groupby('target').gain_logloss.mean().dropna()   # un point par dataset
    if len(d)<5 or d.abs().sum()==0: continue
    tr.append(dict(tranche=str(lab),gain_moyen=round(float(d.mean()),5),
        gain_median=round(float(d.median()),5),positifs=int((d>0).sum()),n=len(d),
        wilcoxon_p=round(float(wilcoxon(d).pvalue),5)))
TR=pd.DataFrame(tr)
if len(TR): display(TR); write_csv(ROOT/'tranches_rapport.csv',TR)

print('\n=== 4. A3 — CORRESPONDANCE STRICTE, ou « au moins aussi long » ? ===')
a3=[]
for cs,g in FT[FT.ft_steps==TOP].groupby('context_size'):
    c=float(g.ctx.mean()); avail=sorted(g.ep_context.unique())
    if len(avail)<2: continue
    matched=min(avail,key=lambda e:abs(np.log2(max(e,1))-np.log2(max(c,1))))
    longer=[e for e in avail if e>matched]
    if not longer: continue
    big=max(longer)
    p=g.pivot_table(index='target',columns='ep_context',values='gain_logloss')
    if matched not in p or big not in p: continue
    d=(p[big]-p[matched]).dropna()
    if len(d)<5 or d.abs().sum()==0: continue
    a3.append(dict(contexte=cs,lignes=round(c,1),episode_apparie=int(matched),
        episode_plus_long=int(big),ecart=round(float(d.mean()),5),
        wilcoxon_p=round(float(wilcoxon(d).pvalue),5),
        victoires_plus_long=int((d>0).sum()),n=len(d)))
A3=pd.DataFrame(a3)
if len(A3):
    A3=A3.sort_values('lignes'); display(A3); write_csv(ROOT/'a3_correspondance.csv',A3)
    tot=int(A3.victoires_plus_long.sum()); nn=int(A3.n.sum())
    print(f'A3 : l épisode plus long gagne {tot}/{nn} comparaisons appariées.')
    print('  -> la correspondance stricte est réfutée : « au moins aussi long » suffit.'
          if tot>nn/2 else
          '  -> la correspondance stricte n est PAS réfutée : c est l autre hypothèse.')

print('\n=== 5. A4 — LE CROISEMENT SE DÉPLACE-T-IL AVEC LA TAILLE D ÉPISODE ? ===')
cr=[]
for (tid,ep),g in FT[FT.ft_steps==TOP].groupby(['target','episode_rows']):
    g=g.dropna(subset=['gain_logloss']).sort_values('ctx')
    pos=g[g.gain_logloss>0]
    if g.empty: continue
    cr.append(dict(target=tid,name=g.name.iloc[0],episode_rows=int(ep),
        dernier_gagnant=(float(pos.ctx.max()) if len(pos) else 0.0),
        gain_au_ref=float(g[g.context_size==str(REF_CONTEXT)].gain_logloss.mean())
                    if (g.context_size==str(REF_CONTEXT)).any() else np.nan))
CR=pd.DataFrame(cr)
if len(CR):
    piv=CR.pivot_table(index='target',columns='episode_rows',values='dernier_gagnant')
    display(piv.round(0)); write_csv(ROOT/'croisements.csv',CR)
    if piv.shape[1]>=2:
        a,b_=piv.columns[0],piv.columns[-1]
        d=(np.log2(piv[b_].clip(lower=1))-np.log2(piv[a].clip(lower=1))).dropna()
        if len(d)>=5 and d.abs().sum()>0:
            print(f'A4 : de {int(a)} à {int(b_)} lignes par épisode, le dernier contexte '
                  f'gagnant se déplace de {d.mean():+.2f} doublement(s) en moyenne, '
                  f'{int((d>0).sum())}/{len(d)} datasets, Wilcoxon p={wilcoxon(d).pvalue:.5f}')

print('\n=== 6. A5 — LA FORME SURVIT-ELLE À UN PAS 10x PLUS PETIT ? ===')
TS=pd.read_csv(TRAIN_PATH) if Path(TRAIN_PATH).exists() else pd.DataFrame()
if LR_PROBE.empty or TS.empty or 'displacement' not in TS.columns:
    print('Axe optimisation non exécuté, ou déplacements absents : rien à comparer.')
else:
    DISP=TS[['target','regime','episode_rows','steps','displacement']].drop_duplicates()
    EPL=sum(LR_EPISODE); ids=sorted(LR_PROBE.target.unique())
    L=LR_PROBE.merge(DISP,left_on=['target','regime','episode_rows','ft_steps'],
                     right_on=['target','regime','episode_rows','steps'],how='left')
    S_=(FT[(FT.episode_rows==EPL)&(FT.target.isin(ids))]
          .merge(DISP,left_on=['target','regime','episode_rows','ft_steps'],
                 right_on=['target','regime','episode_rows','steps'],how='left'))
    both=pd.concat([L,S_],ignore_index=True)
    t5=(both.groupby(['regime','ft_steps'])
            .agg(deplacement_moyen=('displacement','mean'),
                 gain_petit=('gain_logloss',lambda s:np.nan),datasets=('target','nunique'))
            .drop(columns='gain_petit').round(4))
    print(f'Déplacement des poids par régime (épisodes de {EPL} lignes, n={len(ids)}) :')
    display(t5)
    g5=(both.groupby(['regime','ft_steps','context_size'])
            .agg(lignes=('ctx','mean'),gain_moyen=('gain_logloss','mean'),
                 datasets=('target','nunique'))
            .reset_index().sort_values(['regime','ft_steps','lignes']).round(5))
    display(g5); write_csv(ROOT/'axe_optimisation.csv',g5)
    # Comparaison à DÉPLACEMENT apparié : pour chaque jalon à pas faible, on prend le
    # jalon standard dont le déplacement est le plus proche, sur le même dataset.
    pairs=[]
    for (tid,cs),g in both.groupby(['target','context_size']):
        lo=g[g.regime=='lrlo'].dropna(subset=['displacement'])
        st=g[g.regime=='standard'].dropna(subset=['displacement'])
        if lo.empty or st.empty: continue
        for _,r in lo.iterrows():
            j=(st.displacement-r.displacement).abs().idxmin()
            pairs.append(dict(target=tid,context_size=cs,lignes=float(r.ctx),
                pas_faible=int(r.ft_steps),depl_faible=float(r.displacement),
                pas_standard=int(st.loc[j,'ft_steps']),depl_standard=float(st.loc[j,'displacement']),
                gain_faible=float(r.gain_logloss),gain_standard=float(st.loc[j,'gain_logloss'])))
    PR=pd.DataFrame(pairs)
    if len(PR):
        PR['ecart']=PR.gain_faible-PR.gain_standard
        write_csv(ROOT/'axe_optimisation_apparie.csv',PR)
        t6=(PR.groupby('context_size')
              .agg(lignes=('lignes','mean'),gain_faible=('gain_faible','mean'),
                   gain_standard=('gain_standard','mean'),ecart=('ecart','mean'),
                   n=('target','nunique')).sort_values('lignes').round(5))
        print('\nÀ déplacement de poids apparié — positif = le petit pas fait mieux')
        display(t6); write_csv(ROOT/'axe_optimisation_table.csv',t6.reset_index())
        d=PR.groupby('target').ecart.mean().dropna()
        if len(d)>=3 and d.abs().sum()>0:
            print(f'A5 : écart moyen {d.mean():+.5f} sur {len(d)} datasets '
                  f'({int((d>0).sum())} en faveur du petit pas).')
        gr=PR[PR.lignes>=REF_CONTEXT]
        if len(gr):
            print(f'  Au contexte >= {REF_CONTEXT} : petit pas {gr.gain_faible.mean():+.5f}, '
                  f'standard {gr.gain_standard.mean():+.5f}. '
                  +('Le dommage persiste : la loi ne vient pas du pas.'
                    if gr.gain_faible.mean()<0 else
                    'Le dommage disparaît : deux mécanismes, à démêler.'))
    else:
        print('Aucune paire à déplacement comparable : le bras standard de ces datasets '
              'vient de l import et n a pas de déplacement mesuré. Relancer l axe '
              'optimisation sur des datasets entraînés dans cette session.')

## 12. FiguresLa figure 2 est celle de l'article : si les trois courbes tombent l'une sur l'autre quandon les trace contre le rapport, la thèse est faite en une image.

In [ ]:

RAMP=['#8FB4DC','#4B7FBB','#1F4E79']; MARKS=['o','s','^']
ORANGE='#D1793A'; GREY='#6B7280'
plt.rcParams.update({'figure.dpi':160,'savefig.dpi':300,'font.size':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.edgecolor':'#9CA3AF','axes.labelcolor':'#374151',
                     'xtick.color':GREY,'ytick.color':GREY,'text.color':'#1F2937'})
def col(i): return RAMP[min(i,len(RAMP)-1)]
def mark(i): return MARKS[min(i,len(MARKS)-1)]

S=FT[(FT.ft_steps==TOP)&(FT.target.isin(common))] if common else FT[FT.ft_steps==TOP]

# --- Figure 1 : gain vs contexte, une courbe par taille d'épisode
if len(S):
    fig,ax=plt.subplots(figsize=(5.8,4.2)); ax.set_xscale('log')
    ax.axhline(0,color=ORANGE,lw=2.0,zorder=2)
    for i,ep in enumerate(SIZES):
        g=(S[(S.episode_rows==ep)&(S.context_size!='full')]
             .groupby('context_size').agg(x=('ctx','mean'),y=('gain_logloss','mean'),
             sd=('gain_logloss','std'),n=('target','nunique')).sort_values('x'))
        if len(g)<2: continue
        ax.errorbar(g.x,g.y,yerr=g.sd/np.sqrt(g.n.clip(lower=1)),color=col(i),lw=1.9,
                    marker=mark(i),ms=6,capsize=2.5,zorder=3,label=f'épisodes de {int(ep)} lignes')
    ax.set_xlabel('Lignes dans le contexte d inférence (échelle log)')
    ax.set_ylabel('Gain de log-loss')
    ax.set_title(f'Le croisement se déplace-t-il ? (n={S.target.nunique()}, {TOP} pas)',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig1_par_episode.png',bbox_inches='tight'); plt.show()

# --- Figure 2 : l'effondrement contre le rapport
if len(S) and S.episode_rows.nunique()>=2:
    fig,ax=plt.subplots(figsize=(5.8,4.2)); ax.set_xscale('log',base=2)
    ax.axhline(0,color=ORANGE,lw=2.0,zorder=2)
    ax.axvline(1,color='#374151',lw=1.1,ls='--',zorder=1)
    for i,ep in enumerate(SIZES):
        g=S[S.episode_rows==ep].dropna(subset=['gain_logloss'])
        if g.empty: continue
        gb=(g.assign(b=np.round(np.log2(g.ratio)*2)/2)
              .groupby('b').agg(x=('ratio','median'),y=('gain_logloss','mean'),
              sd=('gain_logloss','std'),n=('target','nunique')).sort_values('x'))
        gb=gb[gb.n>=3]           # une moyenne sur 1 ou 2 datasets n'est pas une moyenne
        if not len(gb): continue
        ax.errorbar(gb.x,gb.y,yerr=gb.sd/np.sqrt(gb.n.clip(lower=1)),color=col(i),lw=1.7,
                    marker=mark(i),ms=5.5,capsize=2,alpha=.95,zorder=3,
                    label=f'épisodes de {int(ep)} lignes')
    y0,y1=ax.get_ylim(); ax.set_ylim(y0-0.22*(y1-y0),y1)
    ax.annotate('on déploie plus court\nqu on a entraîné',xy=(1,y0-0.19*(y1-y0)),
                xytext=(-8,0),textcoords='offset points',ha='right',va='bottom',
                fontsize=7.5,color=GREY)
    ax.annotate('on extrapole\nau-delà',xy=(1,y0-0.19*(y1-y0)),xytext=(8,0),
                textcoords='offset points',ha='left',va='bottom',fontsize=7.5,color=GREY)
    ax.set_xlabel('Rapport  contexte d inférence / contexte d entraînement')
    ax.set_ylabel('Gain de log-loss')
    ax.set_title('Une seule courbe, si le rapport est la bonne variable',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig2_effondrement.png',bbox_inches='tight'); plt.show()

# --- Figure 3 : le déplacement du croisement, dataset par dataset
if len(CR) and CR.episode_rows.nunique()>=2:
    piv=CR.pivot_table(index='target',columns='episode_rows',values='dernier_gagnant')
    fig,ax=plt.subplots(figsize=(5.4,3.9)); ax.set_yscale('log')
    for tid,row in piv.iterrows():
        v=row.dropna()
        if len(v)<2: continue
        ax.plot([int(c) for c in v.index],v.clip(lower=16).values,lw=1.0,color=RAMP[1],
                alpha=.35,marker='o',ms=3,mew=0,zorder=2)
    m=piv.median()
    ax.plot([int(c) for c in m.index],m.clip(lower=16).values,lw=2.4,color=RAMP[2],
            marker='s',ms=7,zorder=4,label='médiane')
    ax.set_xscale('log',base=2)
    ax.set_xlabel('Lignes par épisode d entraînement')
    ax.set_ylabel('Dernier contexte où le fine-tuning gagne encore')
    ax.set_title('Entraîner plus long repousse la limite',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig3_croisement.png',bbox_inches='tight'); plt.show()

write_json(ROOT/'RUN_COMPLETE.json',dict(run_hash=RUN_HASH,
    datasets=int(CUR.target.nunique()),arms=sorted(CUR.arm.unique().tolist()),
    episode_sizes=[int(s) for s in SIZES],common_subset=sorted(int(t) for t in common),
    points=len(CUR),excluded=FAILURES,hours_left=round(hours_left(),2),
    note='development study; OpenML outer fold 0 never read'))
print('\nFichiers dans',ROOT)
for f in sorted(ROOT.glob('*.csv'))+sorted(ROOT.glob('*.png'))+sorted(ROOT.glob('*.json')):
    print('  ',f.name)

## Comment lire ce run**Le premier chiffre** est la ligne A3. Si l'épisode plus long bat l'épisode apparié surune majorité nette de comparaisons, la correspondance stricte est morte et la règledevient « entraîner au moins aussi long que le contexte de déploiement ». C'est laréponse directe à ta question, et c'est la seule des deux qui donne une recetteutilisable : un fine-tuning unique, sur les épisodes les plus longs que la mémoiresupporte, valable pour tous les contextes en dessous.**Le deuxième** est la figure 2. La décroissance en fonction du contexte, tu l'as déjà ;elle n'étonne plus personne. Trois courbes issues de trois entraînements différents quitombent l'une sur l'autre dès qu'on les trace contre le rapport, c'est autre chose : çadit que la variable pertinente n'est ni le contexte ni l'épisode, mais leur désaccord. Unrelecteur qui voit cet effondrement n'a plus grand-chose à objecter ; s'il ne se produitpas, la table des tranches de rapport dira franchement que non.**Le troisième** est la table A2. Elle donne le chiffre qu'on cite : à quel rapport legain change de signe. Si c'est autour de 1, la règle est belle. Si c'est autour de 2 ou4, elle est plus tolérante et il faut l'écrire ainsi.**Ce que la limite de mémoire impose, et qui fait partie du résultat.** La tailled'épisode est bornée par la T4 — c'est la raison d'être du plafond de 1024 qui traînaitdans tes notebooks. Si la règle du rapport est vraie, alors il existe un contexte au-delàduquel **aucun** fine-tuning réalisable sur ce matériel ne peut aider. Le run précédentle montrait déjà : à « tout le train », les deux tailles donnaient −0,005. Ce n'est pasun échec de l'expérience, c'est une borne, et elle a sa place dans la discussion.**Les réserves qui restent.** Une seule graine. Un seul taux d'apprentissage. Unfine-tuning épisodique sur la cible elle-même — pas de LoRA, pas d'adaptateur, pas detête légère. Le rapport de requêtes est figé à 0,2 et varie donc en valeur absolue avecla taille d'épisode : un relecteur attentif le remarquera, et la réponse honnête est quece n'est pas testé ici. Écris-le dans les limites plutôt que d'attendre qu'on te ledemande.